# CoffeeGuard — Notebook 02b: rò rỉ theo cấp độ model — **NHE** (`resnet18`)

**Bật GPU:** `Runtime` → `Change runtime type` → `T4 GPU`.

Một trong ba notebook `02b_cap_do_{nhe,trung,nang}`. Ba file **chỉ khác dòng `CAP_DO`**.
Mỗi phiên Colab chạy **một model × ba cách chia** (theo cây, theo lá, theo ảnh) liền một mạch.

| | |
|---|---|
| Model | `resnet18`, 11,2 triệu tham số (đếm bằng `scripts/kiem_cam.py`, đầu 6 lớp) |
| Thời gian ước tính trên T4 | 15–25 phút cho cả 3 cách chia — `ASSUMED`, chưa đo |
| Xuất ra | `reports/predictions_cap_do/pred__*.csv` (3 file) — **chỉ dự đoán, không lưu trọng số** |

### Câu hỏi

Lượt A–C cho thấy chia sai làm macro-F1 phồng +0,5 đến +3,2 điểm nhưng **0/6 cặp đổi kết luận**.
Cả bốn kiến trúc khi đó đều nhẹ, cỡ gần nhau, nên chúng "thuộc lòng" mặt kia của lá ở mức tương
tự. Ở đây chỉ đổi **sức chứa** (cùng họ ResNet, cùng công thức train của lượt A): nếu model lớn
hưởng lợi từ rò rỉ nhiều hơn hẳn, thứ hạng có thể đảo. Kết quả có thể vẫn là "không đảo".

### Hạn chế biết trước

- **Một seed** mỗi cấu hình (như lượt A–C): nhiễu seed không nằm trong CI.
- Công thức train (LR 3e-4, 15 epoch) chỉnh cho model nhẹ. Model lớn có thể chưa ở điểm tối ưu —
  ảnh hưởng **so giữa các cấp**, không ảnh hưởng mức phồng của cùng một model.

### Chống ngắt quãng

- Mỗi cách chia xong → ghi dự đoán → **chép ngay sang Drive** (`ketqua_tam_capdo_resnet18/`).
- Mất kết nối: **Runtime → Run all** lại. Cách chia nào đã có trên Drive thì chép về và bỏ qua.
- Giữ tab mở. Colab miễn phí ngắt khi tab không hoạt động lâu.

## 0. Khởi động

In [ ]:
# --- O KHOI DONG: chay duoc CA tren Colab LAN tren laptop ---
import os, sys, json, time, zipfile, hashlib, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    TREN_COLAB = True
except ImportError:
    TREN_COLAB = False

if TREN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    GOC = Path("/content/CoffeeGuard")
    GOC.mkdir(parents=True, exist_ok=True)
    os.environ["COFFEEGUARD_ROOT"] = str(GOC)

    # Ma nguon hop dong (~40 KB). Tim theo thu tu: da giai nen > Drive > upload.
    if not (GOC / "src" / "coffeeguard" / "bench" / "contract.py").exists():
        nguon = Path("/content/drive/MyDrive/CoffeeGuard/coffeeguard_src.zip")
        if not nguon.exists():
            print("Chon file coffeeguard_src.zip (o §0 cua notebook 01 tren laptop sinh ra):")
            from google.colab import files
            up = files.upload()
            nguon = Path(next(iter(up)))
        with zipfile.ZipFile(nguon) as z:
            z.extractall(GOC)
        print("da giai nen ma nguon ->", GOC / "src")
    SRC = GOC / "src"
else:
    # Notebook nam o <repo>/notebooks/ nen goc repo la thu muc cha.
    GOC = Path.cwd()
    while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
        GOC = GOC.parent
    SRC = GOC / "src"

sys.path.insert(0, str(SRC))

from coffeeguard.bench import contract as ct
from coffeeguard.bench import paths as P

assert P.ROOT == GOC, f"Goc du an khong khop: {P.ROOT} != {GOC}"
print("moi truong :", "Google Colab" if TREN_COLAB else "laptop")
print("goc du an  :", P.ROOT)
print("classes_6  :", list(ct.CLASSES_6))

## 1. Cấu hình

In [ ]:
# ============================ CHINH O DAY ============================
GOI            = "coffeeguard_bench_leafsquare-ngan1.4x_224.zip"
SHA256_GOI     = "98bbfe7c001d6eda5a95a4c7e72fcb1ecdcfa2fcebf246dcce07d9a4452e6454"  # sinh 22/10 (co split ro ri); de "" = BO QUA kiem chung
#                  ^ khop file data/coffeeguard_bench_*.zip.sha256.
#                    Doi goi (dong goi lai) thi PHAI cap nhat dong nay,
#                    khong thi cell §2 dung lai voi loi SHA256 KHONG KHOP.
THU_MUC_DRIVE  = "MyDrive/CoffeeGuard"


# THI NGHIEM RO RI THEO CAP DO MODEL (mo rong G1, Docs/Khoang_trong_bai_bao.md).
# Cung mot ho (ResNet), chi khac do lon -> bien thay doi duy nhat la SUC CHUA.
# Ca 3 qua bo loc CAM bang hanh vi (scripts/kiem_cam.py, timm 1.0.30, 31/10/2026):
#   resnet18 11.18M sai so 1.18e-07 | resnet50 23.52M 6.39e-08 | resnet152 58.16M 6.75e-08
# Ba notebook 02b CHI khac dong CAP_DO duoi day.
CAP_DO = "nhe"   # "nhe" | "trung" | "nang"
_MODEL_THEO_CAP = {
    "nhe":   ("resnet18",  "resnet18"),
    "trung": ("resnet50",  "resnet50"),
    "nang":  ("resnet152", "resnet152"),
}
KIEN_TRUC = [_MODEL_THEO_CAP[CAP_DO]]

# Cong thuc train GIONG HET luot A (notebook 02) — KHONG chinh rieng cho model lon,
# neu khong thi khong con biet chenh lech do suc chua hay do cong thuc.
EPOCHS       = 15
BATCH        = 32
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
SEED         = 42
CHAY_FOLD    = [0, 1, 2, 3, 4]   # chay thu [0] thi KHONG ghi file (moi anh phai vao test dung 1 lan)

CAC_CACH_CHIA = ("theo_cay", "theo_la", "theo_anh")
# CO DINH cho ca 3 notebook: `bench.py ro-ri --cap-do` ghep "<goc>-theo_la" voi "<goc>-theo_cay".
RUN_ID_GOC   = "capdo"
# Chi luu du doan. Model train tren split ro ri KHONG BAO GIO duoc vao app, va luot
# theo cay o day cung khong thay luot A -> khong luu .pt, khong ghi de_export.json.
LUU_PT       = False
# = P.PREDICTIONS_CAP_DO. Ghi chuoi thang vi goi Colab (22/10) mang paths.py cu.
THU_MUC_RA   = "reports/predictions_cap_do"

CACH_CHIA = CAC_CACH_CHIA[0]          # o §2 nap split nay truoc; vong §7 nap lai tung cach
RUN_ID    = f"{RUN_ID_GOC}-{CACH_CHIA}"
# =====================================================================

print("goi    :", GOI)
print("cap do :", CAP_DO, "->", KIEN_TRUC[0][0])
print("chia   :", ", ".join(CAC_CACH_CHIA), "<- THI NGHIEM, khong dung cho app")
print("fold   :", CHAY_FOLD)

## 2. Nạp gói + kiểm chứng

Thứ tự kiểm: file tồn tại → sha256 khớp → giải nén → **thứ tự lớp trong file
split khớp `contract.py`**. Bước cuối dễ bị bỏ qua nhất và có hậu quả tệ nhất.

In [ ]:
def bam(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

if TREN_COLAB:
    nguon = Path("/content/drive") / THU_MUC_DRIVE / GOI
    if not nguon.exists():
        raise FileNotFoundError(
            f"Khong thay {nguon}.\nChay notebook 01 roi tai file .zip len Drive."
        )
    h = bam(nguon)
    print("sha256 goi:", h)
    if SHA256_GOI and h != SHA256_GOI:
        raise ValueError(
            "SHA256 KHONG KHOP — goi tai len hong hoac khong phai ban moi nhat.\n"
            f"  tren dia : {h}\n  mong doi : {SHA256_GOI}\n"
            "DUNG LAI. Train tren goi hong thi moi con so deu vo nghia ma van chay tron tru."
        )
    if not SHA256_GOI:
        print("[CANH BAO] chua dat SHA256_GOI -> khong kiem chung duoc tinh toan ven.")
    # LUON giai nen de (truoc 23/10 bo qua neu da co data/splits): may ao dung lai
    # tu phien truoc se giu src/ va file split CU, va notebook chay tron tru tren do.
    with zipfile.ZipFile(nguon) as z:
        z.extractall(GOC)
    print("da giai nen ->", GOC)

    # O §1 da import coffeeguard tu coffeeguard_src.zip (co the CU) TRUOC khi goi nay
    # giai nen ma nguon moi. Python giu module cu trong bo nho -> nap lai tu dia.
    # Da xay ra 23/10/2026: paths cu thieu PREDICTIONS_RO_RI, splits moi thi co.
    import importlib
    for _ten in ["coffeeguard.bench.paths", "coffeeguard.bench.contract",
                 *sorted(m for m in sys.modules if m.startswith("coffeeguard."))]:
        if _ten in sys.modules:
            importlib.reload(sys.modules[_ten])
    from coffeeguard.bench import paths as P
    from coffeeguard.bench import contract as ct
    assert hasattr(P, "PREDICTIONS_RO_RI"), "ma nguon trong goi van la ban cu"
else:
    print("chay local — dung thang du lieu trong repo, khong can goi .zip")

from coffeeguard.bench import index as bidx
from coffeeguard.bench import splits as bsplit
from coffeeguard.bench import predictions as bpred
from coffeeguard.bench import latency as blat

# tu no assert thu tu lop khop contract; split ro ri bi CHAN neu khong khai bao
meta  = bsplit.doc_split(bsplit.duong_dan_split(CACH_CHIA),
                         cho_phep_ro_ri=(CACH_CHIA != "theo_cay"))
index = bsplit.gan_fold(bidx.doc_index(), meta)

# Chon thu muc anh THEO TEN, khong lay "thu muc dau tien tim thay": co ca bo
# leafsquare lan bo letterbox, lay nham thi van train tron tru tren bo kia.
ten_bo = Path(GOI).stem.replace("coffeeguard_bench_", "")
THU_MUC_ANH = GOC / "data" / "processed" / ten_bo
if not THU_MUC_ANH.exists():
    co = [d.name for d in (GOC / "data" / "processed").iterdir() if d.is_dir()]
    raise FileNotFoundError(f"Khong thay bo anh '{ten_bo}'. Dang co: {co}")
manifest = json.loads((THU_MUC_ANH / "manifest.json").read_text(encoding="utf-8"))

print("\n--- san sang ---")
print("  bo anh     :", THU_MUC_ANH.name, "|", sum(1 for _ in THU_MUC_ANH.glob("*.png")), "PNG")
print("  che do cat :", manifest["che_do_cat"], "| neo", manifest.get("leaf_neo"),
      "| scale", manifest["leaf_scale"])
print("  la chiem   :", manifest["ti_le_pixel_la"])
print("  anh/cay/la :", meta["n_images"], meta["n_plants"], meta["n_leaves"])
print("  split seed :", meta["seed"], "|", meta["thuat_toan"])
print("  cach chia  :", meta.get("cach_chia", "theo_cay"), "| nhom theo", meta["nhom_theo"])
for c in meta["canh_bao"]:
    print("  [CANH BAO]", c)

## 3. Thư viện

In [ ]:
def cai(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# GHIM PHIEN BAN — vi sao:
#   Cau truc DAU model phu thuoc phien ban timm. Bo loc CAM (§4 ke hoach) da duoc
#   kiem chung tren timm 1.0.30; mot ban khac co the doi thu tu lop trong head,
#   lam dang thuc CAM vo ma KHONG bao loi. Colab `pip install timm` lay ban moi
#   nhat, nen phai ghim. O §3b ngay duoi se kiem lai bang HANH VI — ghim chi la
#   lop bao ve thu nhat, phep kiem moi la lop that.
TIMM_GHIM = "timm==1.0.30"

# Truoc 15/10 o nay CHI cai khi chua co timm. Colab co san 1.0.29 nen luot 30/09
# chay 1.0.29 du "ghim" 1.0.30 — chi in canh bao. Nay: doc phien ban bang
# importlib.metadata (KHONG import timm), lech thi cai dung ban roi moi import.
from importlib import metadata
try:
    _timm_co = metadata.version("timm")
except metadata.PackageNotFoundError:
    _timm_co = None
if _timm_co != TIMM_GHIM.split("==")[1]:
    print(f"timm hien co: {_timm_co} -> cai {TIMM_GHIM}")
    cai(TIMM_GHIM)
import timm
assert timm.__version__ == TIMM_GHIM.split("==")[1], (
    f"timm {timm.__version__} van khac ban ghim — timm da duoc import truoc o nay. "
    "Runtime > Restart session roi chay lai tu dau.")
# KHONG cai onnx / onnxruntime tren Colab — CO Y.
#   Colab ghim protobuf 5.29.6, con `onnx` moi duoc build voi gencode 6.31.1
#   -> import la ne VersionError. Nang protobuf len 6.x thi pha `grpcio-status`
#   (ghim <6.0dev) va `google-ai-generativelanguage`. Ha `onnx` xuong ban cu chi
#   la phong doan, co the vo lai khi Colab doi anh nen.
#   => Colab CHI train. Export ONNX lam tren laptop bang `scripts/export_onnx.py`,
#      noi onnx va protobuf da khop nhau. Xem §9.

import torch, numpy as np, pandas as pd
pd.set_option("display.width", 200)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("timm", timm.__version__)

# Ghi lai moi truong vao bien de dong goi ket qua mang ve (truy nguyen duoc)
MOI_TRUONG = {"torch": torch.__version__, "timm": timm.__version__,
              "python": sys.version.split()[0],
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cpu":
    print()
    print("=" * 66)
    print("[CANH BAO] DANG CHAY CPU — KHONG PHAI GPU")
    print("  5 kien truc x 5 fold tren CPU mat NHIEU GIO thay vi 1-2 gio.")
    print("  Runtime > Change runtime type > T4 GPU > Save.")
    print("  LUU Y: doi runtime se KHOI DONG LAI may ao va xoa moi thu da cai,")
    print("         nen hay doi TRUOC roi chay lai notebook tu dau.")
    print("=" * 66)


## 4. Dataset + augmentation

### Vì sao `RandomResizedCrop` chứ không để nguyên

Ảnh trong gói đã là **cửa sổ vuông neo cạnh ngắn 1,4×**, tức lá chiếm ~61% khung
và **hai đầu lá đã bị cắt**. Hai hệ quả, giải cùng một cách:

1. Lúc chạy thật, nông dân được PWA dặn *"đưa lá vào khung"* nên lá chiếm 70%+.
   Model train một tỉ lệ mà chạy ở tỉ lệ khác thì tụt chính xác **mà không có gì
   báo**.
2. Cửa sổ cố định luôn rơi vào **giữa** lá, nên model không bao giờ thấy phần
   gần chóp.

→ `RandomResizedCrop(scale=(0.45, 1.0))` vừa trải rộng tỉ lệ lá quanh khoảng lúc
chạy thật, vừa làm mỗi epoch ô cắt rơi vào một đoạn khác của lá.

### Vì sao lật cả dọc lẫn ngang

Lá cà phê không có chiều "đúng"; ảnh RoCoLe chụp cầm tay, hướng tuỳ ý.

### Vì sao `mean`/`std` lấy theo từng model

Quy tắc 11. ViT và MobileNet chuẩn hoá khác nhau. Dùng chung một pipeline thì
một model nhận đầu vào lệch phân bố, trông kém đi vài phần trăm, và ta kết luận
sai rằng *"kiến trúc này kém"*. Giá trị lấy từ
`timm.data.resolve_model_data_config` và được ghi vào `registry.json` để PWA
dùng **đúng** con số đó.

### Vì sao MỌI model chạy 224, kể cả kiến trúc có kích thước gốc khác

`timm` ghi kích thước gốc 256 cho `mobilevitv2_050/075` và `fastvit_t8`, 192 cho
`tf_efficientnetv2_b0`. Lượt 30/09 đánh giá theo kích thước gốc (phóng PNG 224
lên 256), trong khi export và app chạy 224: **9,3% ảnh fold 4 đổi nhãn** so với
model đã đánh giá, và phép đối chiếu ONNX-vs-PyTorch không thấy gì vì cả hai phía
đều chạy 224 (`reports/kich_thuoc_dau_vao.json`).

→ Train, đánh giá, đo tốc độ và export đều dùng `ct.IMG_SIZE` (224). Hồ sơ ghi
`kich_thuoc_danh_gia`; `export_onnx.py` dừng nếu nó khác kích thước export. Bộ
lọc CAM vẫn kiểm ở kích thước gốc (quy tắc 16) **và** kiểm thêm ở 224.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

LOP2ID = {c: i for i, c in enumerate(ct.CLASSES_6)}

class BoDuLieuLa(Dataset):
    def __init__(self, df_, thu_muc, bien_doi):
        self.png = df_.png.tolist()
        self.y = [LOP2ID[c] for c in df_.label6]
        self.thu_muc = Path(thu_muc)
        self.bien_doi = bien_doi
    def __len__(self):
        return len(self.png)
    def __getitem__(self, i):
        with Image.open(self.thu_muc / self.png[i]) as im:
            return self.bien_doi(im.convert("RGB")), self.y[i]

def bien_doi_train(size, mean, std):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.45, 1.0), ratio=(0.85, 1.18),
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomApply([T.ColorJitter(0.25, 0.25, 0.20, 0.03)], p=0.7),
        T.ToTensor(), T.Normalize(mean, std),
    ])

def bien_doi_test(size, mean, std):
    # KHONG augmentation, KHONG crop, KHONG phong anh. Anh da khoa o ct.IMG_SIZE.
    # Phong len kich thuoc goc cua kien truc = danh gia mot dieu kien app khong
    # bao gio chay (luot 30/09: mobilevitv2_050 danh gia 256, app chay 224).
    if size != ct.IMG_SIZE:
        raise ValueError(f"bien_doi_test({size}) — moi buoc phai dung ct.IMG_SIZE = {ct.IMG_SIZE}")
    return T.Compose([T.ToTensor(), T.Normalize(mean, std)])

def trong_so_lop(df_):
    """1/sqrt(n) — giua 'khong can bang gi' va 'oversample thang'.

    Oversample theo 1/n day rust_level_4 (29 anh) lap lai ~27 lan moi epoch;
    model thuoc long dung 29 tam do, recall tren train vot len con test thi
    khong. 1/sqrt(n) giam bot ma van keo lop hiem ra khoi vung bi bo qua.
    """
    n = df_.label6.value_counts().reindex(list(ct.CLASSES_6)).fillna(0).to_numpy()
    w = 1.0 / np.sqrt(np.maximum(n, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32)

display(pd.crosstab(index.label6, index.fold).reindex(list(ct.CLASSES_6)))

## 5. Model + CAM tính sẵn trong graph

`onnxruntime-web` **chỉ suy luận, không có gradient**, nên Grad-CAM không chạy
được trong trình duyệt. Thay bằng **CAM gốc** (Zhou 2016):

```
cam[k] = conv1x1(feature_map, W_classifier[k])
```

Nhúng phép đó vào graph ONNX lúc export thì JS chỉ việc lấy kênh của lớp được
dự đoán — không phải chuyển trọng số phân loại xuống client.

**Ràng buộc:** kiến trúc phải kết thúc `GAP → một Linear` *và* số kênh feature
map phải bằng `in_features` của Linear đó.

`mobilenetv3` **không thoả**: head của nó là `GAP → conv_head → act → Linear`.
Lớp `ModelCAM` phát hiện, báo rõ, rồi cho model đó chạy tiếp **không có CAM** —
nó vẫn dự thi phần phân loại, chỉ mất bản đồ nhiệt. Đây là **một kết quả của đồ
án**: ràng buộc *"hiển thị trực quan vùng tổn thương"* trong thuyết minh loại
bớt kiến trúc nào.

In [ ]:
import torch.nn as nn
from coffeeguard.bench import cam as bcam


class ModelCAM(nn.Module):
    """timm backbone -> (logits, cam).

    Kiem tinh hop le cua CAM bang HANH VI (`coffeeguard.bench.cam`), khong bang
    ten module hay so kenh. Vi sao: kiem bang so kenh cho DUONG TINH GIA voi ho
    ConvNeXt/EdgeNeXt — so kenh khop, lop cuoi dung la nn.Linear, nhung co chuan
    hoa toan cuc sau pooling nen dang thuc CAM vo. Da do: convnext_tiny sai so
    1.8e-01 ma kiem cau truc van noi "co CAM". Xem cam.py va §4 ke hoach.
    """

    def __init__(self, ten_timm, n_lop, pretrained=True):
        super().__init__()
        self.ten = ten_timm
        self.backbone = timm.create_model(ten_timm, pretrained=pretrained, num_classes=n_lop)
        self.head = self.backbone.get_classifier()
        # kiem_cam TU dua model ve eval() roi tra lai che do cu. Bat buoc, vi
        # timm.create_model tra model o che do TRAIN, va o che do do BatchNorm
        # dung thong ke theo batch (phi tuyen theo dau vao) -> phep thu cong
        # tinh cho AM TINH GIA. Da mac that: mobilevitv2_050 bao 1.4e-01 o che
        # do train nhung 4.0e-08 o eval. Khong truyen kich thuoc -> lay kich
        # thuoc GOC cua kien truc (mobilevitv2_050 chay 256, khong phai 224).
        kq = bcam.kiem_cam(self.backbone)
        # Kiem THEM o ct.IMG_SIZE — kich thuoc that su train, danh gia va export.
        kq_dung = bcam.kiem_cam(self.backbone, ct.IMG_SIZE)
        self.sai_so_cam_224 = kq_dung["sai_so"]
        kq["co_cam"] = kq["co_cam"] and kq_dung["co_cam"]
        self.co_cam = kq["co_cam"]
        self.ly_do = kq["ly_do"]
        self.sai_so_cam = kq["sai_so"]
        self.kich_thuoc_kiem = kq["kich_thuoc"]
        self.cuu_duoc = kq["cuu_duoc"]

    def forward(self, x):
        f = self.backbone.forward_features(x)
        logits = self.backbone.forward_head(f)
        if not self.co_cam:
            return logits
        return logits, torch.einsum("bchw,kc->bkhw", f, self.head.weight)


def tao_model(ten_timm, pretrained=True):
    m = ModelCAM(ten_timm, len(ct.CLASSES_6), pretrained=pretrained)
    cfg = timm.data.resolve_model_data_config(m.backbone)
    return m, {
        "ten_timm": ten_timm,
        # input_size = kich thuoc THAT SU dung; input_size_goc chi de tham khao.
        "input_size": [3, ct.IMG_SIZE, ct.IMG_SIZE],
        "input_size_goc": list(cfg["input_size"]),
        "kich_thuoc_danh_gia": ct.IMG_SIZE,
        "mean": [round(float(v), 6) for v in cfg["mean"]],
        "std":  [round(float(v), 6) for v in cfg["std"]],
        "interpolation": cfg.get("interpolation", "bicubic"),
        "co_cam": m.co_cam,
        "sai_so_cam": m.sai_so_cam,
        "kich_thuoc_kiem": m.kich_thuoc_kiem,
        "sai_so_cam_224": m.sai_so_cam_224,
        "ly_do_khong_cam": m.ly_do,
        **blat.dem_tham_so(m),
    }


# --- CHAN TRUOC KHI TRAIN: tha dung ngay con hon train 5 fold roi moi biet ---
# Nem RuntimeError neu bat ky kien truc nao mat CAM. Day la lop bao ve THAT
# (khac voi viec ghim phien ban timm o §3, chi la lop thu nhat).
_ = bcam.chan_neu_mat_cam([t for t, _b in KIEN_TRUC], len(ct.CLASSES_6))
print("OK — ca", len(KIEN_TRUC), "kien truc deu thoa dang thuc CAM tren timm",
      timm.__version__)

SOI = {}
hang = []
for ten, bi_danh in KIEN_TRUC:
    _m, tt = tao_model(ten, pretrained=False)
    SOI[bi_danh] = tt
    hang.append({"arch": bi_danh, "params_M": tt["params_M"],
                 "size_MB_fp32": tt["size_MB_fp32"], "input": tt["input_size"][1], "goc": tt["input_size_goc"][1],
                 "CAM": "co" if tt["co_cam"] else "KHONG",
                 "sai_so_cam": f"{tt['sai_so_cam']:.1e}" if tt["sai_so_cam"] is not None else "-",
                 "kiem_o": tt["kich_thuoc_kiem"],
                 "ly_do": tt["ly_do_khong_cam"][:50]})
    del _m
display(pd.DataFrame(hang))


## 6. Vòng huấn luyện 5 fold

Mỗi fold: train trên 4 fold còn lại, dự đoán fold này. Mỗi ảnh vào test **đúng
một lần** → gộp lại đủ 1.559 dự đoán, trong đó `rust_level_4` có 29 thay vì 3.

In macro-F1 của từng fold ngay khi xong: fold nào lệch hẳn khỏi các fold còn lại
là dấu hiệu **train hỏng**, không phải "phương sai tự nhiên".

In [ ]:
from sklearn.metrics import f1_score

def dat_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.inference_mode()
def du_doan(model, loader):
    model.eval()
    ra = []
    for x, _ in loader:
        out = model(x.to(DEV, non_blocking=True))
        logits = out[0] if isinstance(out, tuple) else out
        ra.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(ra)

def train_mot_fold(ten_timm, fold, tt, epochs=EPOCHS):
    dat_seed(SEED + fold)
    size = ct.IMG_SIZE
    assert tt["kich_thuoc_danh_gia"] == size, "ho so va hop dong lech kich thuoc"
    tr_df, te_df = index[index.fold != fold], index[index.fold == fold]

    dl_tr = DataLoader(BoDuLieuLa(tr_df, THU_MUC_ANH, bien_doi_train(size, tt["mean"], tt["std"])),
                       batch_size=BATCH, shuffle=True, num_workers=2,
                       pin_memory=True, drop_last=True)
    dl_te = DataLoader(BoDuLieuLa(te_df, THU_MUC_ANH, bien_doi_test(size, tt["mean"], tt["std"])),
                       batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model, _ = tao_model(ten_timm, pretrained=True)
    model.to(DEV)
    lf = nn.CrossEntropyLoss(weight=trong_so_lop(tr_df).to(DEV), label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs,
                                              steps_per_epoch=len(dl_tr), pct_start=0.25)
    scaler = torch.amp.GradScaler(enabled=(DEV == "cuda"))

    t0 = time.time()
    for ep in range(epochs):
        model.train(); tong = 0.0
        for x, y in dl_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEV, enabled=(DEV == "cuda")):
                out = model(x)
                loss = lf(out[0] if isinstance(out, tuple) else out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sch.step()
            tong += loss.item() * x.size(0)
        print(f"    ep{ep+1:02d}/{epochs} loss {tong/len(dl_tr.dataset):.4f}", end="\r")

    Pm = du_doan(model, dl_te)
    y_true = np.array([LOP2ID[c] for c in te_df.label6])
    f1 = f1_score(y_true, Pm.argmax(1), average="macro", zero_division=0)
    print(f"    fold {fold}: macro-F1 = {f1:.4f}   ({time.time()-t0:.0f}s)          ")
    return model, te_df, Pm, f1

def train_toan_bo(ten_timm, bi_danh):
    tt = SOI[bi_danh]
    print(f"\n=== {bi_danh} ({ten_timm}) ===")
    manh, f1s, model_cuoi = [], [], None
    for fold in CHAY_FOLD:
        model, te_df, Pm, f1 = train_mot_fold(ten_timm, fold, tt)
        f1s.append(f1); model_cuoi = model
        sub = te_df[["png", "fold", "plant_id", "leaf_id", "face"]].copy()
        sub["y_true"] = te_df.label6.to_numpy()
        sub["y_pred"] = [ct.CLASSES_6[i] for i in Pm.argmax(1)]
        for j, c in enumerate(ct.CLASSES_6):
            sub[f"p_{c}"] = Pm[:, j]
        manh.append(sub)
        # Luu trong so TUNG FOLD (tu 23/10/2026). Truoc do chi con model fold cuoi
        # -> tren laptop khong do lai duoc gi tren fold 0-3 (vd TTA,
        # scripts/do_tta.py). Thu muc con folds/: export_onnx.py chi doc de_export.json
        # nen khong bao gio export nham model mot fold. Thi nghiem ro ri: KHONG luu.
        if LUU_PT:   # 02b: LUU_PT = False -> khong luu
            d_fold = GOC / "models" / "pt" / "folds"
            d_fold.mkdir(parents=True, exist_ok=True)
            pt_fold = d_fold / f"{bi_danh}_{RUN_ID}_fold{fold}.pt"
            torch.save({k: v.detach().cpu() for k, v in model.backbone.state_dict().items()}, pt_fold)
            ben_hoa(f"models/pt/folds/{pt_fold.name}")
        if fold != CHAY_FOLD[-1]:
            del model; torch.cuda.empty_cache()
    pred = pd.concat(manh, ignore_index=True)
    sd = np.std(f1s, ddof=1) if len(f1s) > 1 else 0.0
    print(f"  macro-F1 tung fold: {[round(x,4) for x in f1s]}")
    print(f"  trung binh {np.mean(f1s):.4f} +/- {sd:.4f}")
    return pred, model_cuoi, f1s

## 7. Chạy 3 cách chia — nối tiếp được sau khi mất kết nối

Mỗi cách chia: nạp lại file split → train 5 fold → `bpred.ghi_du_doan()` kiểm chứng rồi ghi →
chép sang Drive. Chạy lại cả notebook thì cách chia nào đã xong (trên máy ảo hoặc Drive) bị bỏ qua.

In [ ]:
# MOT model x 3 cach chia. Ben hoa sau MOI cach chia: may ao Colab co the bi thu
# hoi bat cu luc nao, Drive thi khong.
import shutil
from sklearn.metrics import f1_score

ten_timm, bi_danh = KIEN_TRUC[0]
RA = GOC / THU_MUC_RA
RA.mkdir(parents=True, exist_ok=True)
SAO_LUU = (Path("/content/drive") / THU_MUC_DRIVE / f"ketqua_tam_capdo_{bi_danh}") if TREN_COLAB else None

TOM_TAT = []
for CACH_CHIA in CAC_CACH_CHIA:
    RUN_ID = f"{RUN_ID_GOC}-{CACH_CHIA}"
    ten = bpred.ten_file("pytorch-timm", bi_danh, RUN_ID)
    dich = RA / ten
    if SAO_LUU is not None and not dich.exists() and (SAO_LUU / ten).exists():
        shutil.copy2(SAO_LUU / ten, dich)
        print(f"[KHOI PHUC] {ten} tu Drive")
    if dich.exists():
        print(f"[DA XONG] {ten} — bo qua {CACH_CHIA}")
        pred = pd.read_csv(dich)
    else:
        # train_mot_fold doc bien TOAN CUC `index` -> phai gan lai cho tung cach chia.
        # Quen buoc nay thi ca 3 lan deu train tren split theo cay ma van tron tru.
        meta = bsplit.doc_split(bsplit.duong_dan_split(CACH_CHIA),
                                cho_phep_ro_ri=(CACH_CHIA != "theo_cay"))
        index = bsplit.gan_fold(bidx.doc_index(), meta)
        assert meta.get("cach_chia", "theo_cay") == CACH_CHIA, "nap nham file split"
        print(f"\n########## {CACH_CHIA} | nhom theo {meta['nhom_theo']} | run_id {RUN_ID}")
        pred, model, _f1s = train_toan_bo(ten_timm, bi_danh)
        del model
        torch.cuda.empty_cache()
        if len(CHAY_FOLD) != meta["n_folds"]:
            print(f"  [CHAY THU {len(CHAY_FOLD)}/{meta['n_folds']} fold] khong ghi file")
            continue
        p = bpred.ghi_du_doan(pred, framework="pytorch-timm", arch=bi_danh,
                              run_id=RUN_ID, thu_muc=RA)
        print(f"  da ghi {p.name}")
        if SAO_LUU is not None:
            try:
                SAO_LUU.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, SAO_LUU / p.name)
                print(f"  da chep sang Drive: {SAO_LUU / p.name}")
            except Exception as e:
                print(f"  [CANH BAO] khong chep duoc sang Drive: {e}")
    TOM_TAT.append({"cach_chia": CACH_CHIA, "n_anh": len(pred),
                    "macro_f1_6lop": round(f1_score(pred.y_true, pred.y_pred,
                                                    average="macro", zero_division=0), 4)})

# Chi de nhin nhanh. So chinh thuc (5 dai, CI theo cay, so theo cap) tinh tren
# laptop bang `python scripts/bench.py ro-ri --cap-do`.
display(pd.DataFrame(TOM_TAT))

## 8. Đóng gói mang về laptop

Zip **chỉ chứa** `reports/predictions_cap_do/` → giải nén đè lên gốc repo không động tới file nào
của lượt A–C.

In [ ]:
ket_qua = Path("/content" if TREN_COLAB else GOC) / f"coffeeguard_ketqua_capdo_{bi_danh}.zip"
cac_file = sorted(RA.glob(f"pred__pytorch-timm__{bi_danh}__{RUN_ID_GOC}-*.csv"))
if len(cac_file) != len(CAC_CACH_CHIA):
    print(f"[CANH BAO] moi co {len(cac_file)}/{len(CAC_CACH_CHIA)} cach chia — chay lai §7 cho du")
with zipfile.ZipFile(ket_qua, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in cac_file:
        z.write(p, f"{THU_MUC_RA}/{p.name}")

if TREN_COLAB:
    try:
        shutil.copy2(ket_qua, Path("/content/drive") / THU_MUC_DRIVE / ket_qua.name)
        print("da chep vao Drive")
    except Exception as e:
        print("khong chep duoc vao Drive:", e)

print(f"{ket_qua}  ({ket_qua.stat().st_size/1048576:.2f} MB)")
print("sha256:", bam(ket_qua))
for n in zipfile.ZipFile(ket_qua).namelist():
    print("   ", n)

## Việc tiếp theo

1. Tải `coffeeguard_ketqua_capdo_*.zip` về, giải nén **đè lên gốc repo**.
2. Khi đủ cả ba notebook (9 file trong `reports/predictions_cap_do/`), trên laptop:

   ```bash
   python scripts/bench.py ro-ri --cap-do
   ```

   Ra `reports/ro_ri_muc_phong__cap_do.csv` (mức phồng từng model),
   `ro_ri_doi_thu_hang__cap_do.csv` (3 cặp có đổi kết luận không) và
   `ro_ri_phoi_nhiem__cap_do.csv`.